In [4]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('./enriched_listings.csv')

print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())

# Selected features (clean distance metrics)
features_for_correlation = [
    'price_total',
    'price_per_bed',
    'sqft',
    'bedrooms',
    'baths',
    'dist_to_memorial_union_mu',                # campus distance
    'dist_to_downtown_davis_3rd_and_g_st',      # downtown distance
    'nearest_grocery_dist',                     # grocery distance
    'has_parking',
    'pets_allowed'
]

# Create correlation dataframe
correlation_df = df[features_for_correlation].dropna().copy()

# Convert boolean columns to numeric
correlation_df['has_parking'] = correlation_df['has_parking'].astype(int)
correlation_df['pets_allowed'] = correlation_df['pets_allowed'].astype(int)

# Convert laundry type to numeric if available
if 'laundry_type' in df.columns:
    laundry_mapping = {
        'on-site': 3,
        'in-unit': 2,
        'off-site': 1,
        'none': 0,
        'N/A': 0
    }

    df['laundry_numeric'] = df['laundry_type'].map(laundry_mapping).fillna(0)
    correlation_df['laundry'] = df.loc[correlation_df.index, 'laundry_numeric']

# Calculate correlation matrix
correlation_matrix = correlation_df.corr()

# Create heatmap
fig = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=correlation_matrix.columns,
    y=correlation_matrix.columns,
    colorscale='Viridis',
    zmid=0,
    colorbar=dict(title="Correlation"),
    text=np.round(correlation_matrix.values, 2),
    texttemplate='%{text}',
    textfont={"size": 9},
    zmin=-1,
    zmax=1
))

fig.update_layout(
    title='Property Features Correlation Matrix',
    template='plotly_white',
    height=750,
    width=850,
    xaxis=dict(tickangle=-45),
    yaxis=dict(autorange='reversed')   # ensures diagonal top-left → bottom-right
)

fig.show()

# Print correlation matrix
print("\nCorrelation Matrix:")
print(correlation_matrix)

print("\nKey Insights:")
print(f"Price vs Square Feet: {correlation_matrix.loc['price_total','sqft']:.3f}")
print(f"Price vs Bedrooms: {correlation_matrix.loc['price_total','bedrooms']:.3f}")
print(f"Price vs Parking: {correlation_matrix.loc['price_total','has_parking']:.3f}")
print(f"Price vs Pets Allowed: {correlation_matrix.loc['price_total','pets_allowed']:.3f}")

if 'laundry' in correlation_matrix.columns:
    print(f"Price vs Laundry: {correlation_matrix.loc['price_total','laundry']:.3f}")

print(f"Price vs Distance to Campus: {correlation_matrix.loc['price_total','dist_to_memorial_union_mu']:.3f}")
print(f"Price vs Distance to Downtown: {correlation_matrix.loc['price_total','dist_to_downtown_davis_3rd_and_g_st']:.3f}")
print(f"Price vs Distance to Grocery: {correlation_matrix.loc['price_total','nearest_grocery_dist']:.3f}")

print(f"Square Feet vs Bedrooms: {correlation_matrix.loc['sqft','bedrooms']:.3f}")

Dataset Shape: (376, 37)

Column Names:
['listing_id', 'complex_name', 'address', 'price_total', 'bedrooms', 'baths', 'sqft', 'lat', 'lon', 'description', 'amenities', 'pets_allowed', 'has_parking', 'laundry_type', 'post_date', 'url', 'neighborhood', 'dist_to_mu', 'price_per_bed', 'price_per_sqft', 'dist_to_memorial_union_mu', 'dist_to_silo', 'dist_to_shields_library', 'dist_to_arc_activities_and_recreation_center', 'dist_to_student_health_center', 'dist_to_trader_joes', 'dist_to_safeway_north', 'dist_to_nugget_markets_east_covell', 'dist_to_davis_food_co-op', 'dist_to_target', 'dist_to_downtown_davis_3rd_and_g_st', 'dist_to_davis_farmers_market', 'dist_to_davis_amtrak_station', 'nearest_grocery', 'nearest_grocery_dist', 'nearest_campus', 'nearest_campus_dist']



Correlation Matrix:
                                     price_total  price_per_bed      sqft  \
price_total                             1.000000      -0.168639  0.643223   
price_per_bed                          -0.168639       1.000000 -0.647243   
sqft                                    0.643223      -0.647243  1.000000   
bedrooms                                0.648375      -0.774871  0.909452   
baths                                   0.467573      -0.548200  0.718437   
dist_to_memorial_union_mu              -0.129509      -0.541505  0.450946   
dist_to_downtown_davis_3rd_and_g_st    -0.170158      -0.563370  0.359324   
nearest_grocery_dist                    0.230914       0.256983 -0.006282   
has_parking                            -0.046089      -0.012557 -0.022222   
pets_allowed                            0.083521       0.049785  0.158808   
laundry                                 0.024479      -0.106034  0.167158   

                                     bedrooms     bath